In [ ]:
import numpy as np, pandas as pd, os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# Experiment 1b — REINFORCE Discrete Channel

Tests the report's untested claim that REINFORCE with control variates closes the discrete accuracy gap while retaining error-detection.

In [ ]:
!pip install torch matplotlib -q

In [ ]:
import os
os.environ['CUDA_VISIBLE_DEVICES'] = ''
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

torch.manual_seed(0); np.random.seed(0)

N_FEATURES, N_CANDIDATES = 16, 10
N_TRAIN, N_VAL = 10000, 2000
VOCAB_SIZE, HIDDEN = 32, 128
N_EPOCHS, BATCH_SIZE, LR = 200, 128, 1e-3
ENTROPY_COEF = 0.05

def make_dataset(n, seed):
    rng = np.random.default_rng(seed)
    objs = rng.integers(0, 2, (n, N_CANDIDATES, N_FEATURES)).astype(np.float32)
    return TensorDataset(torch.tensor(objs[:, 0, :]), torch.tensor(objs),
                         torch.zeros(n, dtype=torch.long))

train_ds = make_dataset(N_TRAIN, 1)
val_ds   = make_dataset(N_VAL, 2)

class Sender(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(N_FEATURES, HIDDEN), nn.ReLU(),
            nn.Linear(HIDDEN, HIDDEN), nn.ReLU(),
            nn.Linear(HIDDEN, VOCAB_SIZE))
    def forward(self, x): return self.net(x)

class ReceiverDiscrete(nn.Module):
    def __init__(self):
        super().__init__()
        self.msg_emb  = nn.Embedding(VOCAB_SIZE, HIDDEN)
        self.obj_proj = nn.Sequential(nn.Linear(N_FEATURES, HIDDEN), nn.ReLU(),
                                      nn.Linear(HIDDEN, HIDDEN))
    def forward(self, tok, cands):
        h = self.msg_emb(tok)
        o = self.obj_proj(cands)
        return torch.bmm(o, h.unsqueeze(-1)).squeeze(-1)

class ReceiverContinuous(nn.Module):
    def __init__(self, msg_dim=VOCAB_SIZE):
        super().__init__()
        self.msg_proj = nn.Sequential(nn.Linear(msg_dim, HIDDEN), nn.ReLU())
        self.obj_proj = nn.Sequential(nn.Linear(N_FEATURES, HIDDEN), nn.ReLU(),
                                      nn.Linear(HIDDEN, HIDDEN))
    def forward(self, msg, cands):
        h = self.msg_proj(msg)
        o = self.obj_proj(cands)
        return torch.bmm(o, h.unsqueeze(-1)).squeeze(-1)

print('Setup complete')

In [ ]:
def train_reinforce(n_epochs=N_EPOCHS):
    sender = Sender(); receiver = ReceiverDiscrete()
    opt = torch.optim.Adam(list(sender.parameters()) + list(receiver.parameters()), lr=LR)
    loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
    baseline = 0.0; history = []
    for epoch in range(n_epochs):
        sender.train(); receiver.train()
        ep_reward = []
        for si, cands, labels in loader:
            logits = sender(si)
            dist = torch.distributions.Categorical(logits=logits)
            tok = dist.sample(); logp = dist.log_prob(tok)
            scores = receiver(tok, cands)
            reward = (scores.argmax(-1) == labels).float()
            advantage = (reward - baseline).detach()
            loss_sender = -(advantage * logp).mean() - ENTROPY_COEF * dist.entropy().mean()
            loss_receiver = F.cross_entropy(scores, labels)
            (loss_sender + loss_receiver).backward()
            opt.step(); opt.zero_grad()
            baseline = 0.95 * baseline + 0.05 * reward.mean().item()
            ep_reward.append(reward.mean().item())
        if (epoch+1) % 40 == 0:
            va = eval_discrete(sender, receiver)
            history.append((epoch+1, va))
            print(f'  REINFORCE ep {epoch+1}: train={np.mean(ep_reward):.3f}  val={va:.3f}')
    return sender, receiver, history

def train_gumbel(n_epochs=N_EPOCHS):
    sender = Sender(); receiver = ReceiverContinuous()
    opt = torch.optim.Adam(list(sender.parameters()) + list(receiver.parameters()), lr=LR)
    loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
    history = []
    for epoch in range(n_epochs):
        tau = 1.0 * (0.1/1.0) ** (epoch/n_epochs)
        sender.train(); receiver.train()
        for si, cands, labels in loader:
            msg = F.gumbel_softmax(sender(si), tau=tau, hard=True)
            loss = F.cross_entropy(receiver(msg, cands), labels)
            loss.backward(); opt.step(); opt.zero_grad()
        if (epoch+1) % 40 == 0:
            va = eval_gumbel(sender, receiver)
            history.append((epoch+1, va))
            print(f'  Gumbel ep {epoch+1}: val={va:.3f}')
    return sender, receiver, history

def train_continuous(n_epochs=N_EPOCHS):
    sender = Sender(); receiver = ReceiverContinuous()
    opt = torch.optim.Adam(list(sender.parameters()) + list(receiver.parameters()), lr=LR)
    loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
    for epoch in range(n_epochs):
        sender.train(); receiver.train()
        for si, cands, labels in loader:
            loss = F.cross_entropy(receiver(torch.tanh(sender(si)), cands), labels)
            loss.backward(); opt.step(); opt.zero_grad()
    return sender, receiver

def eval_discrete(sender, receiver):
    sender.eval(); receiver.eval(); accs = []
    with torch.no_grad():
        for si, cands, labels in DataLoader(val_ds, batch_size=256):
            accs.append((receiver(sender(si).argmax(-1), cands).argmax(-1)==labels).float().mean().item())
    return float(np.mean(accs))

def eval_gumbel(sender, receiver):
    sender.eval(); receiver.eval(); accs = []
    with torch.no_grad():
        for si, cands, labels in DataLoader(val_ds, batch_size=256):
            msg = F.one_hot(sender(si).argmax(-1), VOCAB_SIZE).float()
            accs.append((receiver(msg, cands).argmax(-1)==labels).float().mean().item())
    return float(np.mean(accs))

def eval_continuous(sender, receiver):
    sender.eval(); receiver.eval(); accs = []
    with torch.no_grad():
        for si, cands, labels in DataLoader(val_ds, batch_size=256):
            accs.append((receiver(torch.tanh(sender(si)), cands).argmax(-1)==labels).float().mean().item())
    return float(np.mean(accs))

def noise_test_discrete(sender, receiver, noise_levels, gumbel=False):
    sender.eval(); receiver.eval(); results = []
    rng = np.random.default_rng(7)
    with torch.no_grad():
        for sigma in noise_levels:
            accs, wrong_ents = [], []
            for si, cands, labels in DataLoader(val_ds, batch_size=256):
                tok = sender(si).argmax(-1)
                flip = torch.tensor(rng.random(tok.shape) < sigma)
                rand_tok = torch.tensor(rng.integers(0, VOCAB_SIZE, tok.shape))
                tok_noisy = torch.where(flip, rand_tok, tok)
                if gumbel:
                    msg = F.one_hot(tok_noisy, VOCAB_SIZE).float()
                    scores = receiver(msg, cands)
                else:
                    scores = receiver(tok_noisy, cands)
                probs = F.softmax(scores, dim=-1)
                pred = scores.argmax(-1)
                accs.append((pred == labels).float().mean().item())
                mask = pred != labels
                if mask.sum() > 0:
                    ent = -(probs[mask] * probs[mask].clamp(1e-9).log()).sum(-1)
                    wrong_ents.append(ent.mean().item())
            results.append((np.mean(accs), np.mean(wrong_ents) if wrong_ents else 0.0))
    return results

def noise_test_continuous(sender, receiver, noise_levels):
    sender.eval(); receiver.eval(); results = []
    with torch.no_grad():
        for sigma in noise_levels:
            accs, wrong_ents = [], []
            for si, cands, labels in DataLoader(val_ds, batch_size=256):
                msg = torch.tanh(sender(si)) + torch.randn_like(sender(si)) * sigma
                # msg already computed above -- fix:
                msg = torch.tanh(sender(si))
                msg_noisy = msg + torch.randn_like(msg) * sigma
                scores = receiver(msg_noisy, cands)
                probs = F.softmax(scores, dim=-1)
                pred = scores.argmax(-1)
                accs.append((pred == labels).float().mean().item())
                mask = pred != labels
                if mask.sum() > 0:
                    ent = -(probs[mask] * probs[mask].clamp(1e-9).log()).sum(-1)
                    wrong_ents.append(ent.mean().item())
            results.append((np.mean(accs), np.mean(wrong_ents) if wrong_ents else 0.0))
    return results

print('Training functions defined. Starting runs...')
print('\n=== REINFORCE discrete ===')
s_rl, r_rl, hist_rl = train_reinforce()
print('\n=== Gumbel-Softmax discrete ===')
s_gs, r_gs, hist_gs = train_gumbel()
print('\n=== Continuous ===')
s_co, r_co = train_continuous()

acc_rl = eval_discrete(s_rl, r_rl)
acc_gs = eval_gumbel(s_gs, r_gs)
acc_co = eval_continuous(s_co, r_co)
print(f'\nFinal accuracy: REINFORCE={acc_rl:.3f}  Gumbel={acc_gs:.3f}  Continuous={acc_co:.3f}')

In [ ]:
NOISE = [0.0, 0.05, 0.1, 0.2, 0.3, 0.5, 0.7, 1.0]
rl_noise = noise_test_discrete(s_rl, r_rl, NOISE)
gs_noise = noise_test_discrete(s_gs, r_gs, NOISE, gumbel=True)
co_noise = noise_test_continuous(s_co, r_co, NOISE)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

ax = axes[0]
if hist_rl: ax.plot([h[0] for h in hist_rl], [h[1] for h in hist_rl], 'o-', label='REINFORCE', color='#1D9E75')
if hist_gs: ax.plot([h[0] for h in hist_gs], [h[1] for h in hist_gs], 's-', label='Gumbel', color='#E24B4A')
ax.axhline(acc_co, color='#378ADD', linestyle='--', label='continuous')
ax.set_xlabel('Epoch'); ax.set_ylabel('Val accuracy')
ax.set_title('Does REINFORCE close the discrete accuracy gap?')
ax.legend(); ax.grid(True, alpha=0.3); ax.set_ylim(0, 1.05)

ax = axes[1]
ax.plot(NOISE, [r[0] for r in rl_noise], 'o-', label='REINFORCE', color='#1D9E75')
ax.plot(NOISE, [r[0] for r in gs_noise], 's-', label='Gumbel', color='#E24B4A')
ax.plot(NOISE, [r[0] for r in co_noise], '^-', label='Continuous', color='#378ADD')
ax.set_xlabel('Noise σ'); ax.set_ylabel('Accuracy')
ax.set_title('Accuracy under channel noise'); ax.legend(); ax.grid(True, alpha=0.3)

ax = axes[2]
ax.plot(NOISE, [r[1] for r in rl_noise], 'o-', label='REINFORCE', color='#1D9E75')
ax.plot(NOISE, [r[1] for r in gs_noise], 's-', label='Gumbel', color='#E24B4A')
ax.plot(NOISE, [r[1] for r in co_noise], '^-', label='Continuous', color='#378ADD')
ax.axhline(np.log(N_CANDIDATES), color='gray', linestyle=':', alpha=0.6, label='max entropy')
ax.set_xlabel('Noise σ'); ax.set_ylabel('Wrong-answer entropy (nats)')
ax.set_title('Error-detection signature\n(rising = knows when it is wrong)')
ax.legend(); ax.grid(True, alpha=0.3)

plt.suptitle('Exp 1b — REINFORCE Discrete Channel\nDoes a better estimator close the accuracy gap while keeping error detection?',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('exp1b_reinforce_discrete.png', dpi=150, bbox_inches='tight')
plt.show()

rl_ent_rise = rl_noise[-1][1] - rl_noise[0][1]
co_ent_rise = co_noise[-1][1] - co_noise[0][1]
print(f'\nAccuracy: REINFORCE={acc_rl:.3f}  Gumbel={acc_gs:.3f}  Continuous={acc_co:.3f}')
print(f'Wrong-answer entropy rise (σ=0→1): REINFORCE={rl_ent_rise:+.3f}  Continuous={co_ent_rise:+.3f}')
print('\nReport claim: REINFORCE with control variates closes the discrete accuracy gap')
print('while error-detection (rising wrong-answer entropy) is retained.')
if acc_rl > acc_gs + 0.03 and rl_ent_rise > 0.15:
    print('VERDICT: CONFIRMED')
elif acc_rl > acc_gs:
    print('VERDICT: PARTIAL — accuracy improved, check entropy rise magnitude')
else:
    print('VERDICT: NOT CONFIRMED — try more epochs')